In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")


In [2]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000194CEA4FC80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000194CEB949B0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

result=model.invoke(messages)

In [4]:
result

AIMessage(content='Bonjour, comment vas‑tu\u202f?', additional_kwargs={'reasoning_content': 'User says: "Hello How are you?" The instruction: translate from English to French. So we need to translate that phrase. Likely "Bonjour, comment ça va ?" Or "Bonjour, comment vas-tu ?" But we want a polite, formal? The phrase "Hello How are you?" in English. In French: "Bonjour, comment vas-tu ?" or "Bonjour, comment allez-vous ?" The user didn\'t specify formality. But typical translation: "Bonjour, comment ça va ?" That is informal. But we can choose "Bonjour, comment allez-vous ?" That\'s polite. Let\'s provide both? The instruction: translate the following from English to French. So we should produce the French translation. Let\'s produce a simple one: "Bonjour, comment vas-tu ?" That is casual. Or "Bonjour, comment allez-vous ?" That is formal. Since the original is informal, I\'d choose informal: "Bonjour, comment vas-tu ?" I\'ll answer that.'}, response_metadata={'token_usage': {'complet

In [5]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Bonjour, comment vas‑tu\u202f?'

In [6]:
### Using LCEL- chain the components
chain=model|parser
chain.invoke(messages)

'Bonjour, comment allez‑vous?'

In [10]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Translate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)



In [11]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [12]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [15]:
##Chaining together components with LCEL
chain=prompt|model|parser
chain.invoke({"language":"French","text":"My name is Shrey?"})

"Je m'appelle Shrey?"